# Cyclones Data Processing (2010–2025)

This notebook prepares the NOAA IBTrACS global tropical cyclone data for subsequent exploratory analysis. The workflow imports the original dataset, selects observations from 2010 to 2025, keeps only the relevant variables, converts the data to the appropriate types, handles missing values, performs quality checks, and saves the result to a new CSV file. The analysis year is derived from `ISO_TIME`, rather than from the season indicated in `SEASON`; the original dataset is not modified.

## 1. File paths

This block imports the required libraries and defines the paths to the original CSV file and the processed CSV file. The subsequent steps can therefore read and save the data using centralized path variables.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
RAW_PATH = ROOT / "datasets" / "raw" / "cyclones-since-1980.csv"
OUTPUT_PATH = ROOT / "datasets" / "processed" / "cyclones_2010_2025.csv"

## 2. Load the original dataset

The code reads the NOAA dataset from the CSV file, skips the second row containing measurement units, and initially loads all columns as text. It also disables automatic conversion of empty values so that the basin code `NA` (North Atlantic) is preserved correctly. Finally, it displays the first rows to verify that the data was loaded.

In [5]:
df = pd.read_csv(
    RAW_PATH,
    skiprows=[1],
    dtype=str,
    keep_default_na=False,
    low_memory=False,
    encoding="utf-8-sig",
)
df.columns = df.columns.str.strip()

display(df.head())

,SID,SEASON,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,...,BOM_GUST_PER,REUNION_GUST,REUNION_GUST_PER,USA_SEAHGT,USA_SEARAD_NE,USA_SEARAD_SE,USA_SEARAD_SW,USA_SEARAD_NW,STORM_SPEED,STORM_DIR
0,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 00:00:00,TS,-12.5,172.5,...,,,,,,,,,6,350
1,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 03:00:00,TS,-12.2,172.4,...,,,,,,,,,6,350
2,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 06:00:00,TS,-11.9,172.4,...,,,,,,,,,5,360
3,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 09:00:00,TS,-11.7,172.4,...,,,,,,,,,4,10
4,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 12:00:00,TS,-11.5,172.5,...,,,,,,,,,4,20


## 3. Filter observations from 2010 to 2025

The code converts `ISO_TIME` to a datetime value and keeps only observations from January 1, 2010 through December 31, 2025. The year is then extracted from the date and stored in the `YEAR` column. It prints the number of observations and unique cyclones and displays the distribution of observations by year. The filter uses the actual observation date, so it may include tracks whose season belongs to a different year.

In [7]:
df["ISO_TIME"] = pd.to_datetime(
    df["ISO_TIME"].astype("string").str.strip(),
    errors="coerce"
)

START_DATE = pd.Timestamp("2010-01-01")
END_DATE = pd.Timestamp("2026-01-01")

df = df.loc[
    (df["ISO_TIME"] >= START_DATE) & (df["ISO_TIME"] < END_DATE)
].copy()

df["YEAR"] = df["ISO_TIME"].dt.year

print(f"Filtered observations: {len(df):,}")
print(f"Unique storms: {df['SID'].nunique():,}")
display(df.groupby("YEAR").size().rename("observations").to_frame())

Filtered observations: 100,949
Unique storms: 1,667


,observations
YEAR,
2010,4906
2011,5612
2012,6273
2013,5958
2014,6060
2015,7298
2016,5882
2017,5672
2018,7713


## 4. Select analysis variables

This block defines the variables needed for the analysis, including identifiers, location, intensity, pressure, speed, and storm direction. It keeps only the columns that are actually available in the dataset, avoiding errors if a variable is missing, and prints the final list of selected columns.

In [8]:
selected_columns = [
    "SID", "SEASON", "YEAR", "NUMBER", "BASIN", "SUBBASIN", "NAME",
    "ISO_TIME", "NATURE", "LAT", "LON", "TRACK_TYPE", "WMO_AGENCY",
    "WMO_WIND", "WMO_PRES", "USA_WIND", "USA_PRES", "USA_SSHS",
    "DIST2LAND", "LANDFALL", "STORM_SPEED", "STORM_DIR",
]

available_columns = [column for column in selected_columns if column in df.columns]
df = df.loc[:, available_columns].copy()

print(f"Selected columns: {len(available_columns)}")
print(available_columns)

Selected columns: 22
['SID', 'SEASON', 'YEAR', 'NUMBER', 'BASIN', 'SUBBASIN', 'NAME', 'ISO_TIME', 'NATURE', 'LAT', 'LON', 'TRACK_TYPE', 'WMO_AGENCY', 'WMO_WIND', 'WMO_PRES', 'USA_WIND', 'USA_PRES', 'USA_SSHS', 'DIST2LAND', 'LANDFALL', 'STORM_SPEED', 'STORM_DIR']


## 5. Convert data types and handle missing values

The code converts quantitative variables to numeric values and cleans categorical variables by removing extra spaces and replacing empty strings with missing values. It removes only rows without a storm identifier, date, or geographic coordinates because these fields are essential; observations with missing wind or pressure measurements are retained. Finally, it calculates the percentage of missing values in each column.

In [9]:
numeric_columns = [
    "SEASON", "NUMBER", "LAT", "LON", "WMO_WIND", "WMO_PRES",
    "USA_WIND", "USA_PRES", "USA_SSHS", "DIST2LAND", "LANDFALL",
    "STORM_SPEED", "STORM_DIR",
]
categorical_columns = [
    "SID", "BASIN", "SUBBASIN", "NAME", "NATURE", "TRACK_TYPE", "WMO_AGENCY"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column].str.strip(), errors="coerce")

for column in categorical_columns:
    if column in df.columns:
        df[column] = df[column].str.strip().replace("", pd.NA)

initial_rows = len(df)
df = df.dropna(subset=["SID", "ISO_TIME", "LAT", "LON"])

print(f"Removed rows with invalid identifiers, dates, or coordinates: {initial_rows - len(df):,}")
display(
    df.isna().mean().mul(100).sort_values(ascending=False)
    .round(2).rename("missing_percent").to_frame()
)

Removed rows with invalid identifiers, dates, or coordinates: 0


,missing_percent
WMO_WIND,56.51
WMO_PRES,51.32
WMO_AGENCY,50.99
USA_PRES,14.89
USA_WIND,14.87
LANDFALL,1.65
SID,0.00
SEASON,0.00
STORM_SPEED,0.00
DIST2LAND,0.00


## 6. Quality checks

This step sorts observations by storm and date, checks for multiple observations with the same identifier and timestamp, summarizes the track types (`TRACK_TYPE`), and displays the final dataset shape, number of cyclones, and date range. These results help verify that the data preparation is consistent before exporting the file.

In [10]:
df = df.sort_values(["SID", "ISO_TIME"]).reset_index(drop=True)

duplicates = df.duplicated(subset=["SID", "ISO_TIME"], keep=False).sum()

print(f"Repeated storm timestamps: {duplicates:,}")

display(
    df["TRACK_TYPE"]
    .value_counts(dropna=False)
    .rename("observations")
    .to_frame()
)

print(f"Final shape: {df.shape}")
print(f"Unique storms: {df['SID'].nunique():,}")
print(f"Date range: {df['ISO_TIME'].min()} to {df['ISO_TIME'].max()}")

Repeated storm timestamps: 0


,observations
TRACK_TYPE,
main,95943
US-PROVISIONAL,4877
PROVISIONAL,86
US-PROVISIONAL_spur,25
spur-merge,18


Final shape: (100949, 22)
Unique storms: 1,667
Date range: 2010-01-01 12:00:00 to 2025-12-31 21:00:00


## 7. Export the processed dataset

The code saves the cleaned DataFrame to the output CSV file without including the index and uses a consistent date format. It prints the file path and size and displays the final rows for a last check of the result.

In [11]:
df.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d %H:%M:%S")

print(f"Saved to: {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MiB")

display(df.head())

Saved to: /Users/fexh/Documents/github/tropical-cyclone-analysis/datasets/processed/cyclones_2010_2025.csv
File size: 11.48 MiB


,SID,SEASON,YEAR,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,...,WMO_AGENCY,WMO_WIND,WMO_PRES,USA_WIND,USA_PRES,USA_SSHS,DIST2LAND,LANDFALL,STORM_SPEED,STORM_DIR
0,2010002S09096,2010,2010,1,SI,WA,EDZANI,2010-01-01 12:00:00,DS,-9.2,...,NaN,NaN,NaN,15.0,1010.0,-3,910,910.0,7,210
1,2010002S09096,2010,2010,1,SI,WA,EDZANI,2010-01-01 15:00:00,DS,-9.5,...,NaN,NaN,NaN,20.0,1007.0,-3,950,950.0,7,215
2,2010002S09096,2010,2010,1,SI,WA,EDZANI,2010-01-01 18:00:00,TS,-9.8,...,NaN,NaN,NaN,25.0,1004.0,-1,990,990.0,7,225
3,2010002S09096,2010,2010,1,SI,WA,EDZANI,2010-01-01 21:00:00,TS,-10.0,...,NaN,NaN,NaN,25.0,1004.0,-1,1024,1024.0,7,235
4,2010002S09096,2010,2010,1,SI,WA,EDZANI,2010-01-02 00:00:00,TS,-10.2,...,NaN,NaN,NaN,25.0,1004.0,-1,1059,1059.0,7,245
